In [ ]:
import os
import glob
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import time

In [ ]:
train_dir = "/content/drive/MyDrive/DeepFake Detection Project/Datasets/Batch_size_8/Train/"
validation_dir = "/content/drive/MyDrive/DeepFake Detection Project/Datasets/Batch_size_8/Validation/"
test_dir = "/content/drive/MyDrive/DeepFake Detection Project/Datasets/Batch_size_8/Test/"

In [ ]:
train_results = pd.read_excel("/content/drive/MyDrive/DeepFake Detection Project/Results/2PI_TDCNN_ConvLSTM/Train_Metrics.xlsx")
validation_results = pd.read_excel("/content/drive/MyDrive/DeepFake Detection Project/Results/2PI_TDCNN_ConvLSTM/Validation_Metrics.xlsx")

In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras.models import Model
from keras.layers import (
    Input, Conv2D, MaxPooling2D, MaxPooling1D, Flatten, LSTM, ConvLSTM2D , Dense, TimeDistributed, Dropout,
    GlobalAveragePooling2D, Concatenate, AveragePooling2D, BatchNormalization, Average,
    Reshape, AveragePooling3D, MaxPooling3D, GlobalMaxPooling3D
)

In [ ]:
def build_1PTCSL(input_shape=(10, 256, 256, 3), hidden_dim=64, num_classes=1):

    in1 = Input(shape=input_shape, name="Input_Layer")
    cnn = tf.keras.Sequential([
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        MaxPooling2D(pool_size=(2, 2)),
    ], name="CNN_Module")


    x = TimeDistributed(cnn, name="Time_Distributed_CNN")(in1)
    x = ConvLSTM2D(hidden_dim, (3, 3), padding='same', return_sequences=True, name="ConvLSTM_Layer", activation='tanh')(x)
    x = GlobalMaxPooling3D()(x)
    x = Flatten(name="Flatten_Layer")(x)
    x = Dense(128, activation='relu', name="Dense_Layer_1")(x)
    x = Dropout(0.5, name="Dropout_Layer")(x)
    x = Dense(num_classes, activation='sigmoid', name="Output_Layer")(x)
    model = Model(inputs=in1, outputs=x, name="1PTCSL")
    model.summary()

    return model

In [ ]:
def build_2PTCSL(input_shape=(10, 256, 256, 3), hidden_dim=64, num_classes=1):

    in1 = Input(shape=input_shape, name="Input_Layer_1")
    in2 = Input(shape=input_shape, name="Input_Layer_2")

    cnn = tf.keras.Sequential([
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        MaxPooling2D(pool_size=(2, 2))
    ], name="CNN_Module")

    x1 = TimeDistributed(cnn, name="Time_Distributed_CNN_1")(in1)
    x2 = TimeDistributed(cnn, name="Time_Distributed_CNN_2")(in2)

    x = Average(name="Averaging_Layer")([x1, x2])

    x = ConvLSTM2D(hidden_dim, (3, 3), padding='same', return_sequences=True, name="ConvLSTM_Layer", activation='tanh')(x)
    x = GlobalMaxPooling3D()(x)
    x = Flatten(name="Flatten_Layer")(x)
    x = Dense(128, activation='relu', name="Dense_Layer_1")(x)
    x = Dropout(0.5, name="Dropout_Layer")(x)
    x = Dense(num_classes, activation='sigmoid', name="Output_Layer")(x)
    model = Model(inputs=[in1, in2], outputs=x, name="2PTCSL")
    model.summary()
    return model

In [ ]:
# Build & Compile Model Once
model = build_1PTCSL(input_shape=(10,256,256,3))
model.save_weights("/content/drive/MyDrive/DeepFake Detection Project/Results/1PI_TDCNN_ConvLSTM/model.weights.h5")
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy','AUC','precision','recall']
)

Model: "1PTCSL"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Input_Layer (InputLayer)        │ (None, 10, 256, 256,   │             0 │
│                                 │ 3)                     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Time_Distributed_CNN            │ (None, 10, 32, 32,     │        93,248 │
│ (TimeDistributed)               │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ConvLSTM_Layer (ConvLSTM2D)     │ (None, 10, 32, 32, 64) │       442,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling3d_25         │ (None, 64)             │             0 │
│ (GlobalMaxPooling3D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Flatten_Layer (Flatten)         │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dense_Layer_1 (Dense)           │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dropout_Layer (Dropout)         │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output_Layer (Dense)            │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 544,321 (2.08 MB)

 Trainable params: 544,321 (2.08 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Build & Compile Model Once
model = build_2PTCSL(input_shape=(10,256,256,3))
model.save_weights("/content/drive/MyDrive/DeepFake Detection Project/Results/2PI_TDCNN_ConvLSTM/model.weights.h5")
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy','AUC','precision','recall']
)

Model: "2PTCSL"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Input_Layer_1       │ (None, 10, 256,   │          0 │ -                 │
│ (InputLayer)        │ 256, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Input_Layer_2       │ (None, 10, 256,   │          0 │ -                 │
│ (InputLayer)        │ 256, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Time_Distributed_C… │ (None, 10, 32,    │     93,248 │ Input_Layer_1[0]… │
│ (TimeDistributed)   │ 32, 128)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Time_Distributed_C… │ (None, 10, 32,    │     93,248 │ Input_Layer_2[0]… │
│ (TimeDistributed)   │ 32, 128)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Averaging_Layer     │ (None, 10, 32,    │          0 │ Time_Distributed… │
│ (Average)           │ 32, 128)          │            │ Time_Distributed… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ConvLSTM_Layer      │ (None, 10, 32,    │    442,624 │ Averaging_Layer[… │
│ (ConvLSTM2D)        │ 32, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 64)        │          0 │ ConvLSTM_Layer[0… │
│ (GlobalMaxPooling3… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Flatten_Layer       │ (None, 64)        │          0 │ global_max_pooli… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dense_Layer_1       │ (None, 128)       │      8,320 │ Flatten_Layer[0]… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dropout_Layer       │ (None, 128)       │          0 │ Dense_Layer_1[0]… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Output_Layer        │ (None, 1)         │        129 │ Dropout_Layer[0]… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 544,321 (2.08 MB)

 Trainable params: 544,321 (2.08 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Early‐Stopping Setup
best_val_loss = np.inf      # best validation loss seen so far
patience = 8                # how many epochs to wait after no improvement
wait = 0                    # counter for epochs with no improvement


for i in range(0,100):
    model = build_2PTCSL(input_shape=(10,256,256,3))
    model.load_weights(f"/content/drive/MyDrive/DeepFake Detection Project/Results/2PI_TDCNN_ConvLSTM/model.weights.h5")
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy','AUC','precision','recall']
    )
    epoch_loss = []
    epoch_accuracy = []
    epoch_auc = []
    epoch_precision = []
    epoch_recall = []
    start_time = time.time()

    # Training Loop
    for j in range(0, 105):
        X0 = np.load(train_dir+f"Batch{j}X0.npy")
        print(f"Mounted Batch{j}X0")
        X1 = np.load(train_dir+f"Batch{j}X1.npy")
        print(f"Mounted Batch{j}X1")
        X2 = np.load(train_dir+f"Batch{j}X2.npy")
        print(f"Mounted Batch{j}X2")
        y = np.load(train_dir+f"Batch{j}y.npy")
        print(f"Mounted Batch{j}y")
        print(f"Epoch {i} of {100} ::: Training Batch {j+1} of {105}")
        history = model.fit([X0, X1], y, epochs=1, verbose=0)

        # Collect training metrics
        epoch_loss.append(history.history['loss'][0])
        epoch_accuracy.append(history.history['accuracy'][0])
        epoch_auc.append(history.history['AUC'][0])
        epoch_precision.append(history.history['precision'][0])
        epoch_recall.append(history.history['recall'][0])
        print(f"Batch {i+1} Loss: {history.history['loss'][0]}; Accuracy: {history.history['accuracy'][0]}; AUC: {history.history['AUC'][0]}; Precision: {history.history['precision'][0]}; Recall: {history.history['recall'][0]} \n")

    # Validation Loop
    val_epoch_loss = []
    val_epoch_accuracy = []
    val_epoch_auc = []
    val_epoch_precision = []
    val_epoch_recall = []

    for j in range(0, 13):
        val_X0 = np.load(validation_dir+f"Batch{j}X0.npy")
        val_X1 = np.load(validation_dir+f"Batch{j}X1.npy")
        val_X2 = np.load(validation_dir+f"Batch{j}X2.npy")
        val_y = np.load(validation_dir+f"Batch{j}y.npy")
        print(f"Epoch {i} of {100} ::: Validation Batch {j+1} of 5")
        val_metrics = model.evaluate([val_X0, val_X2], val_y, verbose=0)

        # Collect validation metrics
        val_epoch_loss.append(val_metrics[0])
        val_epoch_accuracy.append(val_metrics[1])
        val_epoch_auc.append(val_metrics[2])
        val_epoch_precision.append(val_metrics[3])
        val_epoch_recall.append(val_metrics[4])

    end_time = time.time()
    elapsed_time = end_time - start_time

    # Average the training metrics for the epoch
    loss_per_epoch = np.mean(epoch_loss)
    accuracy_per_epoch = np.mean(epoch_accuracy)
    auc_per_epoch = np.mean(epoch_auc)
    precision_per_epoch = np.mean(epoch_precision)
    recall_per_epoch = np.mean(epoch_recall)

    metrics = {
        'Loss': loss_per_epoch,
        'Accuracy': accuracy_per_epoch,
        'AUC': auc_per_epoch,
        'Precision': precision_per_epoch,
        'Recall': recall_per_epoch
    }
    train_results = pd.concat([train_results, pd.DataFrame([metrics])], ignore_index=True)
    train_results.to_excel("/content/drive/MyDrive/DeepFake Detection Project/Results/2PI_TDCNN_ConvLSTM/Train_Metrics.xlsx", index=False)

    # Average the validation metrics for the epoch
    val_loss_per_epoch = np.mean(val_epoch_loss)
    val_accuracy_per_epoch = np.mean(val_epoch_accuracy)
    val_auc_per_epoch = np.mean(val_epoch_auc)
    val_precision_per_epoch = np.mean(val_epoch_precision)
    val_recall_per_epoch = np.mean(val_epoch_recall)

    metrics = {
        'Loss': val_loss_per_epoch,
        'Accuracy': val_accuracy_per_epoch,
        'AUC': val_auc_per_epoch,
        'Precision': val_precision_per_epoch,
        'Recall': val_recall_per_epoch
    }
    validation_results = pd.concat([validation_results, pd.DataFrame([metrics])], ignore_index=True)
    validation_results.to_excel("/content/drive/MyDrive/DeepFake Detection Project/Results/2PI_TDCNN_ConvLSTM/Validation_Metrics.xlsx", index=False)

    # Print metrics for the epoch
    print(f"\nEpoch {i} completed in {elapsed_time:.2f} seconds")
    print(f"Training - Loss: {loss_per_epoch:.4f}, Accuracy: {accuracy_per_epoch:.4f}, "
          f"AUC: {auc_per_epoch:.4f}, Precision: {precision_per_epoch:.4f}, Recall: {recall_per_epoch:.4f}")
    print(f"Validation - Loss: {val_loss_per_epoch:.4f}, Accuracy: {val_accuracy_per_epoch:.4f}, "
          f"AUC: {val_auc_per_epoch:.4f}, Precision: {val_precision_per_epoch:.4f}, Recall: {val_recall_per_epoch:.4f}")

    # Early Stopping Check
    if val_loss_per_epoch < best_val_loss - 1e-3:
        best_val_loss = val_loss_per_epoch
        wait = 0
        # save best weights
        model.save_weights(
            "/content/drive/MyDrive/DeepFake Detection Project/Results/2PI_TDCNN_ConvLSTM/best_model.weights.h5"
        )
    else:
        wait += 1
        if wait >= patience:
            print(f"\nEarly stopping at epoch {i+1}. "
                  f"Restoring best model from epoch {i+1-wait}.")
            model.load_weights(
                "/content/drive/MyDrive/DeepFake Detection Project/Results/2PI_TDCNN_ConvLSTM/best_model.weights.h5"
            )
            break

    # Save checkpoint every epochs
    model.save_weights(f"/content/drive/MyDrive/DeepFake Detection Project/Results/2PI_TDCNN_ConvLSTM/model.weights.h5")

Model: "2PTCSL"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Input_Layer_1       │ (None, 10, 256,   │          0 │ -                 │
│ (InputLayer)        │ 256, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Input_Layer_2       │ (None, 10, 256,   │          0 │ -                 │
│ (InputLayer)        │ 256, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Time_Distributed_C… │ (None, 10, 32,    │     93,248 │ Input_Layer_1[0]… │
│ (TimeDistributed)   │ 32, 128)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Time_Distributed_C… │ (None, 10, 32,    │     93,248 │ Input_Layer_2[0]… │
│ (TimeDistributed)   │ 32, 128)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Averaging_Layer     │ (None, 10, 32,    │          0 │ Time_Distributed… │
│ (Average)           │ 32, 128)          │            │ Time_Distributed… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ConvLSTM_Layer      │ (None, 10, 32,    │    442,624 │ Averaging_Layer[… │
│ (ConvLSTM2D)        │ 32, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 64)        │          0 │ ConvLSTM_Layer[0… │
│ (GlobalMaxPooling3… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Flatten_Layer       │ (None, 64)        │          0 │ global_max_pooli… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dense_Layer_1       │ (None, 128)       │      8,320 │ Flatten_Layer[0]… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dropout_Layer       │ (None, 128)       │          0 │ Dense_Layer_1[0]… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Output_Layer        │ (None, 1)         │        129 │ Dropout_Layer[0]… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 544,321 (2.08 MB)

 Trainable params: 544,321 (2.08 MB)

 Non-trainable params: 0 (0.00 B)

Mounted Batch0X0
Mounted Batch0X1
Mounted Batch0X2
Mounted Batch0y
Epoch 0 of 100 ::: Training Batch 1 of 105
Batch 1 Loss: 0.6823289394378662; Accuracy: 0.625; AUC: 0.625; Precision: 0.0; Recall: 0.0 

Mounted Batch1X0
Mounted Batch1X1
Mounted Batch1X2
Mounted Batch1y
Epoch 0 of 100 ::: Training Batch 2 of 105
Batch 1 Loss: 0.673879086971283; Accuracy: 0.75; AUC: 0.1666666567325592; Precision: 0.0; Recall: 0.0 

Mounted Batch2X0
Mounted Batch2X1
Mounted Batch2X2
Mounted Batch2y
Epoch 0 of 100 ::: Training Batch 3 of 105
Batch 1 Loss: 0.6862433552742004; Accuracy: 0.625; AUC: 0.4000000059604645; Precision: 0.0; Recall: 0.0 

Mounted Batch3X0
Mounted Batch3X1
Mounted Batch3X2
Mounted Batch3y
Epoch 0 of 100 ::: Training Batch 4 of 105
Batch 1 Loss: 0.6223872303962708; Accuracy: 0.625; AUC: 0.699999988079071; Precision: 0.0; Recall: 0.0 

Mounted Batch4X0
Mounted Batch4X1
Mounted Batch4X2
Mounted Batch4y
Epoch 0 of 100 ::: Training Batch 5 of 105
Batch 1 Loss: 0.9136034846305847; Accuracy

... (130 outputs left out)

 Total params: 544,321 (2.08 MB)

 Trainable params: 544,321 (2.08 MB)

 Non-trainable params: 0 (0.00 B)

Mounted Batch0X0
Mounted Batch0X1
Mounted Batch0X2
Mounted Batch0y
Epoch 22 of 100 ::: Training Batch 1 of 105
Batch 23 Loss: 0.5154046416282654; Accuracy: 0.75; AUC: 0.8333332538604736; Precision: 0.5; Recall: 1.0 

Mounted Batch1X0
Mounted Batch1X1
Mounted Batch1X2
Mounted Batch1y
Epoch 22 of 100 ::: Training Batch 2 of 105
Batch 23 Loss: 0.3788158595561981; Accuracy: 0.75; AUC: 0.8333332538604736; Precision: 0.5; Recall: 0.5 

Mounted Batch2X0
Mounted Batch2X1
Mounted Batch2X2
Mounted Batch2y
Epoch 22 of 100 ::: Training Batch 3 of 105
Batch 23 Loss: 0.5434870719909668; Accuracy: 0.75; AUC: 0.8666666746139526; Precision: 0.6666666865348816; Recall: 0.6666666865348816 

Mounted Batch3X0
Mounted Batch3X1
Mounted Batch3X2
Mounted Batch3y
Epoch 22 of 100 ::: Training Batch 4 of 105
Batch 23 Loss: 0.3530609607696533; Accuracy: 0.875; AUC: 1.0; Precision: 1.0; Recall: 0.6666666865348816 

Mounted Batch4X0
Mounted Batch4X1
Mounted Batch4X2
Mounted Batch4y
Epoch 22 of 100 ::: Training Batch

In [ ]:
train_results = pd.read_excel("/content/drive/MyDrive/DeepFake Detection Project/Results/1PI_TDCNN_ConvLSTM/Train_Metrics.xlsx")
validation_results = pd.read_excel("/content/drive/MyDrive/DeepFake Detection Project/Results/1PI_TDCNN_ConvLSTM/Validation_Metrics.xlsx")

In [ ]:
# Early‐Stopping Setup
best_val_loss = np.inf      # best validation loss seen so far
patience = 8                # how many epochs to wait after no improvement
wait = 0                    # counter for epochs with no improvement


for i in range(0,100):
    model = build_1PTCSL(input_shape=(10,256,256,3))
    model.load_weights(f"/content/drive/MyDrive/DeepFake Detection Project/Results/1PI_TDCNN_ConvLSTM/model.weights.h5")
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy','AUC','precision','recall']
    )
    epoch_loss = []
    epoch_accuracy = []
    epoch_auc = []
    epoch_precision = []
    epoch_recall = []
    start_time = time.time()

    # Training Loop
    for j in range(0, 105):
        X0 = np.load(train_dir+f"Batch{j}X0.npy")
        print(f"Mounted Batch{j}X0")
        X1 = np.load(train_dir+f"Batch{j}X1.npy")
        print(f"Mounted Batch{j}X1")
        X2 = np.load(train_dir+f"Batch{j}X2.npy")
        print(f"Mounted Batch{j}X2")
        y = np.load(train_dir+f"Batch{j}y.npy")
        print(f"Mounted Batch{j}y")
        print(f"Epoch {i} of {100} ::: Training Batch {j+1} of {105}")
        history = model.fit(X0, y, epochs=1, verbose=0)

        # Collect training metrics
        epoch_loss.append(history.history['loss'][0])
        epoch_accuracy.append(history.history['accuracy'][0])
        epoch_auc.append(history.history['AUC'][0])
        epoch_precision.append(history.history['precision'][0])
        epoch_recall.append(history.history['recall'][0])
        print(f"Batch {i+1} Loss: {history.history['loss'][0]}; Accuracy: {history.history['accuracy'][0]}; AUC: {history.history['AUC'][0]}; Precision: {history.history['precision'][0]}; Recall: {history.history['recall'][0]} \n")

    # Validation Loop
    val_epoch_loss = []
    val_epoch_accuracy = []
    val_epoch_auc = []
    val_epoch_precision = []
    val_epoch_recall = []

    for j in range(0, 13):
        val_X0 = np.load(validation_dir+f"Batch{j}X0.npy")
        val_X1 = np.load(validation_dir+f"Batch{j}X1.npy")
        val_X2 = np.load(validation_dir+f"Batch{j}X2.npy")
        val_y = np.load(validation_dir+f"Batch{j}y.npy")
        print(f"Epoch {i} of {100} ::: Validation Batch {j+1} of 5")
        val_metrics = model.evaluate(val_X0, val_y, verbose=0)

        # Collect validation metrics
        val_epoch_loss.append(val_metrics[0])
        val_epoch_accuracy.append(val_metrics[1])
        val_epoch_auc.append(val_metrics[2])
        val_epoch_precision.append(val_metrics[3])
        val_epoch_recall.append(val_metrics[4])

    end_time = time.time()
    elapsed_time = end_time - start_time

    # Average the training metrics for the epoch
    loss_per_epoch = np.mean(epoch_loss)
    accuracy_per_epoch = np.mean(epoch_accuracy)
    auc_per_epoch = np.mean(epoch_auc)
    precision_per_epoch = np.mean(epoch_precision)
    recall_per_epoch = np.mean(epoch_recall)

    metrics = {
        'Loss': loss_per_epoch,
        'Accuracy': accuracy_per_epoch,
        'AUC': auc_per_epoch,
        'Precision': precision_per_epoch,
        'Recall': recall_per_epoch
    }
    train_results = pd.concat([train_results, pd.DataFrame([metrics])], ignore_index=True)
    train_results.to_excel("/content/drive/MyDrive/DeepFake Detection Project/Results/1PI_TDCNN_ConvLSTM/Train_Metrics.xlsx", index=False)

    # Average the validation metrics for the epoch
    val_loss_per_epoch = np.mean(val_epoch_loss)
    val_accuracy_per_epoch = np.mean(val_epoch_accuracy)
    val_auc_per_epoch = np.mean(val_epoch_auc)
    val_precision_per_epoch = np.mean(val_epoch_precision)
    val_recall_per_epoch = np.mean(val_epoch_recall)

    metrics = {
        'Loss': val_loss_per_epoch,
        'Accuracy': val_accuracy_per_epoch,
        'AUC': val_auc_per_epoch,
        'Precision': val_precision_per_epoch,
        'Recall': val_recall_per_epoch
    }
    validation_results = pd.concat([validation_results, pd.DataFrame([metrics])], ignore_index=True)
    validation_results.to_excel("/content/drive/MyDrive/DeepFake Detection Project/Results/1PI_TDCNN_ConvLSTM/Validation_Metrics.xlsx", index=False)

    # Print metrics for the epoch
    print(f"\nEpoch {i} completed in {elapsed_time:.2f} seconds")
    print(f"Training - Loss: {loss_per_epoch:.4f}, Accuracy: {accuracy_per_epoch:.4f}, "
          f"AUC: {auc_per_epoch:.4f}, Precision: {precision_per_epoch:.4f}, Recall: {recall_per_epoch:.4f}")
    print(f"Validation - Loss: {val_loss_per_epoch:.4f}, Accuracy: {val_accuracy_per_epoch:.4f}, "
          f"AUC: {val_auc_per_epoch:.4f}, Precision: {val_precision_per_epoch:.4f}, Recall: {val_recall_per_epoch:.4f}")

    # Early Stopping Check
    if val_loss_per_epoch < best_val_loss - 1e-3:
        best_val_loss = val_loss_per_epoch
        wait = 0
        # save best weights
        model.save_weights(
            "/content/drive/MyDrive/DeepFake Detection Project/Results/1PI_TDCNN_ConvLSTM/best_model.weights.h5"
        )
    else:
        wait += 1
        if wait >= patience:
            print(f"\nEarly stopping at epoch {i+1}. "
                  f"Restoring best model from epoch {i+1-wait}.")
            model.load_weights(
                "/content/drive/MyDrive/DeepFake Detection Project/Results/1PI_TDCNN_ConvLSTM/best_model.weights.h5"
            )
            break

    # Save checkpoint every epochs
    model.save_weights(f"/content/drive/MyDrive/DeepFake Detection Project/Results/1PI_TDCNN_ConvLSTM/model.weights.h5")

Model: "1PTCSL"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Input_Layer (InputLayer)        │ (None, 10, 256, 256,   │             0 │
│                                 │ 3)                     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Time_Distributed_CNN            │ (None, 10, 32, 32,     │        93,248 │
│ (TimeDistributed)               │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ConvLSTM_Layer (ConvLSTM2D)     │ (None, 10, 32, 32, 64) │       442,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling3d_27         │ (None, 64)             │             0 │
│ (GlobalMaxPooling3D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Flatten_Layer (Flatten)         │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dense_Layer_1 (Dense)           │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dropout_Layer (Dropout)         │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output_Layer (Dense)            │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 544,321 (2.08 MB)

 Trainable params: 544,321 (2.08 MB)

 Non-trainable params: 0 (0.00 B)

Mounted Batch0X0
Mounted Batch0X1
Mounted Batch0X2
Mounted Batch0y
Epoch 0 of 100 ::: Training Batch 1 of 105
Batch 1 Loss: 0.7039037346839905; Accuracy: 0.375; AUC: 0.2083333283662796; Precision: 0.0; Recall: 0.0 

Mounted Batch1X0
Mounted Batch1X1
Mounted Batch1X2
Mounted Batch1y
Epoch 0 of 100 ::: Training Batch 2 of 105
Batch 1 Loss: 0.6669856905937195; Accuracy: 0.75; AUC: 0.75; Precision: 0.0; Recall: 0.0 

Mounted Batch2X0
Mounted Batch2X1
Mounted Batch2X2
Mounted Batch2y
Epoch 0 of 100 ::: Training Batch 3 of 105
Batch 1 Loss: 0.6803431510925293; Accuracy: 0.625; AUC: 0.4333333373069763; Precision: 0.0; Recall: 0.0 

Mounted Batch3X0
Mounted Batch3X1
Mounted Batch3X2
Mounted Batch3y
Epoch 0 of 100 ::: Training Batch 4 of 105
Batch 1 Loss: 0.6449921727180481; Accuracy: 0.625; AUC: 0.7333333492279053; Precision: 0.0; Recall: 0.0 

Mounted Batch4X0
Mounted Batch4X1
Mounted Batch4X2
Mounted Batch4y
Epoch 0 of 100 ::: Training Batch 5 of 105
Batch 1 Loss: 0.99383944272995; Accuracy:

... (76 outputs left out)

 Total params: 544,321 (2.08 MB)

 Trainable params: 544,321 (2.08 MB)

 Non-trainable params: 0 (0.00 B)

Mounted Batch0X0
Mounted Batch0X1
Mounted Batch0X2
Mounted Batch0y
Epoch 13 of 100 ::: Training Batch 1 of 105
Batch 14 Loss: 0.40236589312553406; Accuracy: 0.75; AUC: 1.0; Precision: 0.5; Recall: 1.0 

Mounted Batch1X0
Mounted Batch1X1
Mounted Batch1X2
Mounted Batch1y
Epoch 13 of 100 ::: Training Batch 2 of 105
Batch 14 Loss: 0.21817001700401306; Accuracy: 0.875; AUC: 1.0; Precision: 0.6666666865348816; Recall: 1.0 

Mounted Batch2X0
Mounted Batch2X1
Mounted Batch2X2
Mounted Batch2y
Epoch 13 of 100 ::: Training Batch 3 of 105
Batch 14 Loss: 0.12818928062915802; Accuracy: 1.0; AUC: 1.0; Precision: 1.0; Recall: 1.0 

Mounted Batch3X0
Mounted Batch3X1
Mounted Batch3X2
Mounted Batch3y
Epoch 13 of 100 ::: Training Batch 4 of 105
Batch 14 Loss: 0.2529281973838806; Accuracy: 0.75; AUC: 1.0; Precision: 1.0; Recall: 0.3333333432674408 

Mounted Batch4X0
Mounted Batch4X1
Mounted Batch4X2
Mounted Batch4y
Epoch 13 of 100 ::: Training Batch 5 of 105
Batch 14 Loss: 0.45540350675582886; Accuracy: 0.